# 🤖 Agent 2 — Bot Detection with RNN (GRU)

**Project:** Social Threat Intelligence  
**Agent:** 2 — Bot Detection  
**Model:** Gated Recurrent Unit (GRU) via PyTorch  
**Dataset:** `bot_detection_data.csv` — 50,000 tweets with bot labels

---
### ✅ How to use this notebook on Google Colab (standalone — no repo needed)
1. Upload **only** `bot_detection_data.csv` to the **Files panel** (left sidebar 📁).
2. Run all cells in order: **Runtime → Run all** (Ctrl+F9).
3. All folders are created automatically — you don't need to upload anything else.
4. At the end, download the 3 model files and copy them to your local `models/` folder.

## 0. Setup — Create folders & install packages

In [ ]:
import os

# Create all required directories so no 'path not found' errors occur
os.makedirs('outputs',            exist_ok=True)   # plots saved here
os.makedirs('models/saved',       exist_ok=True)   # model weights
os.makedirs('models/vectorizers', exist_ok=True)   # scaler + meta

print('✅ Folders created:')
for d in ['outputs', 'models/saved', 'models/vectorizers']:
    print(f'   {d}/')

In [ ]:
# PyTorch is pre-installed on Colab; install remaining packages
!pip install scikit-learn pandas numpy matplotlib seaborn joblib --quiet
print('✅ Packages ready.')

In [ ]:
import math
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

## 1. Load Dataset

In [ ]:
# ── Option A: file already uploaded to Colab Files panel ─────────────────────
CSV_PATH = 'bot_detection_data.csv'

# ── Option B: upload interactively (uncomment if Option A doesn't work) ───────
# from google.colab import files
# uploaded = files.upload()          # a dialog will appear
# CSV_PATH = list(uploaded.keys())[0]

# ── Option C: load from Google Drive ─────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# CSV_PATH = '/content/drive/MyDrive/bot_detection_data.csv'

df_raw = pd.read_csv(CSV_PATH)
print(f'✅ Loaded {len(df_raw):,} rows')
print('Columns:', list(df_raw.columns))
df_raw.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
print('Shape:', df_raw.shape)
print('\nMissing values:')
print(df_raw.isnull().sum())
print('\nBot Label distribution:')
print(df_raw['Bot Label'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Class distribution
df_raw['Bot Label'].value_counts().plot(
    kind='bar', ax=axes[0], color=['steelblue', 'tomato'], rot=0
)
axes[0].set_title('Bot Label Distribution')
axes[0].set_xticklabels(['Human (0)', 'Bot (1)'])

# Follower count by label
df_raw.boxplot(column='Follower Count', by='Bot Label', ax=axes[1])
axes[1].set_title('Follower Count by Label')

# Retweet count by label
df_raw.boxplot(column='Retweet Count', by='Bot Label', ax=axes[2])
axes[2].set_title('Retweet Count by Label')

plt.tight_layout()
# Save to 'outputs/' folder (always exists — created in cell 0)
plt.savefig('outputs/eda_bot_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Plot saved to outputs/eda_bot_distribution.png')

## 3. Preprocessing & Feature Engineering

In [ ]:
def clean_tweet(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def engineer_features(df):
    df = df.copy()
    df['tweet_clean']   = df['Tweet'].apply(clean_tweet)
    df['tweet_length']  = df['tweet_clean'].apply(len)
    df['word_count']    = df['tweet_clean'].apply(lambda x: len(x.split()))
    df['Hashtags']      = df['Hashtags'].fillna('')
    df['hashtag_count'] = df['Hashtags'].apply(lambda x: len(x.split()) if x.strip() else 0)
    df['has_hashtag']   = (df['hashtag_count'] > 0).astype(int)
    df['verified_int']  = df['Verified'].astype(int)
    dt = pd.to_datetime(df['Created At'], errors='coerce')
    df['hour_of_day']   = dt.dt.hour.fillna(-1).astype(int)
    df['day_of_week']   = dt.dt.dayofweek.fillna(-1).astype(int)
    df['month']         = dt.dt.month.fillna(-1).astype(int)
    df['is_weekend']    = (df['day_of_week'] >= 5).astype(int)
    df['is_night']      = ((df['hour_of_day'] >= 22) | (df['hour_of_day'] <= 6)).astype(int)
    return df

df = df_raw.drop_duplicates().dropna(subset=['Bot Label'])
df = engineer_features(df)
print(f'✅ After preprocessing: {len(df):,} rows')
df.head()

In [ ]:
# Features used by the RNN
RNN_FEATURES = [
    'Retweet Count', 'Mention Count', 'Follower Count',
    'verified_int', 'tweet_length', 'word_count',
    'hashtag_count', 'has_hashtag',
    'hour_of_day', 'day_of_week', 'month', 'is_weekend', 'is_night',
]
SEQ_LEN = 5

X_raw = df[RNN_FEATURES].fillna(0).values.astype(np.float32)
y     = df['Bot Label'].astype(int).values

print(f'Feature matrix: {X_raw.shape}')
print(f'Labels: {y.shape}, classes: {np.unique(y)}')

## 4. Train / Validation / Test Split + Scaling

In [ ]:
# 70% train  |  15% val  |  15% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_raw, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.176, random_state=42, stratify=y_train_val
)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

# StandardScaler fitted ONLY on training data (no data leakage)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)
print('✅ Scaler fitted on training data only.')

## 5. Build Sequences for RNN

In [ ]:
def build_sequences(X, seq_len=SEQ_LEN):
    """Reshape (n, features) → (n, seq_len, step_size) for RNN input."""
    n, n_feat = X.shape
    step_size = math.ceil(n_feat / seq_len)
    pad = step_size * seq_len - n_feat
    if pad > 0:
        X = np.hstack([X, np.zeros((n, pad))])
    return X.reshape(n, seq_len, step_size).astype(np.float32)

X_train_seq = build_sequences(X_train_sc)
X_val_seq   = build_sequences(X_val_sc)
X_test_seq  = build_sequences(X_test_sc)

INPUT_SIZE = X_train_seq.shape[2]
print(f'Sequence shapes — train: {X_train_seq.shape}, val: {X_val_seq.shape}, test: {X_test_seq.shape}')
print(f'input_size (per time step): {INPUT_SIZE}')

In [ ]:
# Build PyTorch DataLoaders
BATCH_SIZE = 256

def to_loader(X_seq, y_arr, batch_size=BATCH_SIZE, shuffle=True):
    dataset = TensorDataset(
        torch.tensor(X_seq, dtype=torch.float32),
        torch.tensor(y_arr, dtype=torch.long),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

train_loader = to_loader(X_train_seq, y_train, shuffle=True)
val_loader   = to_loader(X_val_seq,   y_val,   shuffle=False)
test_loader  = to_loader(X_test_seq,  y_test,  shuffle=False)

print(f'✅ Batches per epoch — train: {len(train_loader)}, val: {len(val_loader)}')

## 6. Define the GRU Model

In [ ]:
class BotRNN(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.3, num_classes=2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.gru = nn.GRU(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.gru(x, h0)
        out = self.dropout(out[:, -1, :])
        return self.fc(out)

model = BotRNN(input_size=INPUT_SIZE).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'✅ Total parameters: {total_params:,}')

## 7. Training

In [ ]:
EPOCHS   = 20
LR       = 1e-3
PATIENCE = 4   # early stopping patience

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)

history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_acc     = 0.0
best_epoch       = 0
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits   = model(xb)
            val_loss += criterion(logits, yb).item()
            correct  += (logits.argmax(1) == yb).sum().item()
            total    += yb.size(0)
    val_loss /= len(val_loader)
    val_acc   = correct / total

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    scheduler.step(val_acc)

    print(f'Epoch {epoch:02d}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | '
          f'Val Loss: {val_loss:.4f} | '
          f'Val Acc: {val_acc:.4f}')

    # ── Early stopping & best-model checkpoint ───────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        best_epoch       = epoch
        patience_counter = 0
        torch.save(model.state_dict(), 'outputs/bot_rnn_best.pth')  # temp checkpoint
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch}. Best val acc: {best_val_acc:.4f} (epoch {best_epoch})')
            break

print(f'\n✅ Training done. Best val accuracy: {best_val_acc:.4f} at epoch {best_epoch}.')

## 8. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs_ran = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs_ran, history['train_loss'], label='Train Loss', marker='o')
ax1.plot(epochs_ran, history['val_loss'],   label='Val Loss',   marker='s')
ax1.set_title('Loss Curves')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(epochs_ran, history['val_acc'], label='Val Accuracy', color='green', marker='^')
ax2.set_title('Validation Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.savefig('outputs/training_curves.png', dpi=120)   # saved to outputs/
plt.show()
print('✅ Plot saved to outputs/training_curves.png')

## 9. Evaluation on Test Set

In [ ]:
# Load best model weights
model.load_state_dict(torch.load('outputs/bot_rnn_best.pth', map_location=DEVICE))
model.eval()

all_preds, all_proba, all_labels = [], [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb    = xb.to(DEVICE)
        logits = model(xb)
        proba  = torch.softmax(logits, dim=1).cpu().numpy()
        preds  = np.argmax(proba, axis=1)
        all_proba.extend(proba[:, 1].tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(yb.numpy().tolist())

print('=== Test Set Evaluation ===')
print(f'Accuracy : {accuracy_score(all_labels, all_preds):.4f}')
print(f'Precision: {precision_score(all_labels, all_preds, zero_division=0):.4f}')
print(f'Recall   : {recall_score(all_labels, all_preds, zero_division=0):.4f}')
print(f'F1 Score : {f1_score(all_labels, all_preds, zero_division=0):.4f}')
print()
print(classification_report(all_labels, all_preds, target_names=['Human', 'Bot'], zero_division=0))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Human', 'Bot'], yticklabels=['Human', 'Bot'])
plt.title('Confusion Matrix — Bot Detection RNN')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('outputs/confusion_matrix_bot.png', dpi=120)   # saved to outputs/
plt.show()
print('✅ Plot saved to outputs/confusion_matrix_bot.png')

## 10. Save Model & Artefacts

In [ ]:
# Save final model weights (models/saved/ already created in cell 0)
torch.save(model.state_dict(), 'models/saved/bot_rnn_model.pth')
print('✅ Model weights saved  → models/saved/bot_rnn_model.pth')

# Save scaler
joblib.dump(scaler, 'models/vectorizers/bot_scaler.joblib')
print('✅ Scaler saved         → models/vectorizers/bot_scaler.joblib')

# Save input metadata (input_size + seq_len) needed by BotDetector
meta = {'input_size': INPUT_SIZE, 'seq_len': SEQ_LEN, 'features': RNN_FEATURES}
joblib.dump(meta, 'models/vectorizers/bot_input_meta.joblib')
print('✅ Input meta saved     → models/vectorizers/bot_input_meta.joblib')

In [ ]:
# Download all 3 model files to your computer
from google.colab import files

files.download('models/saved/bot_rnn_model.pth')
files.download('models/vectorizers/bot_scaler.joblib')
files.download('models/vectorizers/bot_input_meta.joblib')

print('✅ Downloads started!')
print()
print('Place the downloaded files in your local repo:')
print('  models/saved/bot_rnn_model.pth')
print('  models/vectorizers/bot_scaler.joblib')
print('  models/vectorizers/bot_input_meta.joblib')

## 11. Quick Inference Test (Integration Contract)

Verifies the output format Agent 3 expects:
```python
{"bot_score": float, "bot_class": int, "bot_label": str}
```

In [ ]:
def predict_sample(model, scaler, sample_dict, rnn_features=RNN_FEATURES, seq_len=SEQ_LEN, device=DEVICE):
    model.eval()
    x    = np.array([[sample_dict.get(f, 0) for f in rnn_features]], dtype=np.float32)
    x_sc = scaler.transform(x)

    step_size = math.ceil(len(rnn_features) / seq_len)
    pad = step_size * seq_len - len(rnn_features)
    if pad > 0:
        x_sc = np.hstack([x_sc, np.zeros((1, pad))])
    x_seq = x_sc.reshape(1, seq_len, step_size)

    tensor = torch.tensor(x_seq, dtype=torch.float32).to(device)
    with torch.no_grad():
        proba = torch.softmax(model(tensor), dim=1).cpu().numpy()[0]

    bot_score = float(proba[1])
    bot_class = int(np.argmax(proba))
    return {
        'bot_score': round(bot_score, 4),
        'bot_class': bot_class,
        'bot_label': 'BOT' if bot_class == 1 else 'HUMAN',
    }

# Example: suspicious bot-like account
sample_bot = {
    'Retweet Count': 999, 'Mention Count': 50, 'Follower Count': 10,
    'verified_int': 0, 'tweet_length': 15, 'word_count': 3,
    'hashtag_count': 10, 'has_hashtag': 1,
    'hour_of_day': 3, 'day_of_week': 6, 'month': 11, 'is_weekend': 1, 'is_night': 1,
}

# Example: normal human account
sample_human = {
    'Retweet Count': 5, 'Mention Count': 1, 'Follower Count': 500,
    'verified_int': 1, 'tweet_length': 80, 'word_count': 14,
    'hashtag_count': 1, 'has_hashtag': 1,
    'hour_of_day': 14, 'day_of_week': 2, 'month': 6, 'is_weekend': 0, 'is_night': 0,
}

r1 = predict_sample(model, scaler, sample_bot)
r2 = predict_sample(model, scaler, sample_human)

print('Bot-like sample  :', r1)
print('Human-like sample:', r2)

# Validate contract format
for r in [r1, r2]:
    assert 'bot_score' in r and 'bot_class' in r and 'bot_label' in r
    assert 0.0 <= r['bot_score'] <= 1.0
    assert r['bot_label'] in ('BOT', 'HUMAN')

print('\n✅ Integration contract format is valid — ready for Agent 3!')